# Collection Lineage via ATBD — Missing-Version Repair

**Skill:** [`Skills/Collection_Lineage_via_ATBD_Skill.md`](../Skills/Collection_Lineage_via_ATBD_Skill.md)
(confidence rubric adapted from [`Collection_Lineage_CSV_Export_Skill.md`](../Skills/Collection_Lineage_CSV_Export_Skill.md))
**Input:** [`metadataUpdates/collections_missing_versionOct22.json`](../metadataUpdates/collections_missing_versionOct22.json)
(produced by `metadataUpdates/find_schema_issues.py`)
**Output:** `output/missing_version_*` (fixes, repaired `MetadataAssociations`, review CSV, Step 8 report)

Every association in the input names a related collection but has **no Version**. Under the new rules the
association is identified by **ShortName** and **Version is mandatory**, so each one needs a CMR-verified version.
The notebook follows the skill's steps and records the evidence behind every choice:

| Skill step | What the notebook does | Function(s) |
|---|---|---|
| 1 — Collection metadata & ATBD URLs | live CMR record of each target; ATBD URLs by skill priority | `fetch_details`, `extract_atbd_urls_from_metadata` |
| 2 — Fetch & validate the ATBD | download, extract text, 5-criteria validation | `fetch_document`, `validate_atbd_for_lineage` |
| 3–4 — Lineage keywords & sections | quote naming the related collection, keyword priority, section/page | `extract_lineage_statements`, `documented_versions` |
| 5 — Lineage graph | the repaired association (Type, ShortName, Version, Description) | `repair_association`, `build_association` |
| 6 — Validate against CMR | exists, not the target itself, level progression, version choice | `resolve_identifier`, `choose_version`, `validate_against_cmr` |
| 7 — Derived collections | self-references of NRT products → standard counterpart / documented companion | `standard_counterpart`, `documented_companion` |
| 8 — Report | fixes JSON, review CSV, Markdown report, confidence | `assess_confidence`, `lineage_report_md` |

**Identifier resolution for this file** (`resolve_identifier`):
1. exact / case-insensitive ShortName lookup in CMR (the common case — 134 of 139 associations);
2. version embedded in the identifier (`VNP46A1_2` → `VNP46A1` v2), checked against CMR;
3. typo correction within the provider (`SENIINEL-1C_SLC` → `SENTINEL-1C_SLC`);
4. otherwise `not_in_cmr`.

> The output `umm_c_patch` holds only `ShortName`, `Version` and the repaired `MetadataAssociations` array.
> Merge that array into the collection's full native UMM-C record before ingesting.

## Configuration

In [1]:
import os

INPUT_FILE = os.path.join("..", "metadataUpdates", "collections_missing_versionOct22.json")
OUTPUT_DIR = "output"
OUTPUT_PREFIX = "missing_version"

ASSOCIATION_ID_FIELD = "ShortName"   # new rule; use "EntryId" to emit UMM-C <= 1.18.x associations
USE_LIVE_CMR = True                  # merge fixes into the live MetadataAssociations array
FETCH_DOCUMENTS = True               # download ATBDs / user guides (cached in output/doc_cache)
MAX_DOCS_PER_COLLECTION = 2          # highest-priority documents per target
MAX_DOC_BYTES = 25_000_000
HTTP_TIMEOUT = 60
FUZZY_CUTOFF = 0.9                   # difflib similarity for a typo correction
TITLE_MATCH_CUTOFF = 0.8             # not used by this notebook (ShortName mismatch only)
REMOVE_UNRESOLVED = False            # True drops unresolved associations from the patch instead of keeping them
LIMIT_RECORDS = None                 # e.g. 5 for a quick test

## Setup — imports, HTTP session, GraphQL helper

`pypdf` is optional. Without it, a standard-library PDF text extractor (zlib + `Tj`/`TJ` operators) is used, which works for most
NASA ATBD PDFs but loses page numbers and some text. For better quote extraction: `pip install pypdf`.

In [2]:
import json, re, io, csv, time, zlib, hashlib, difflib
import html as htmllib
from collections import Counter, defaultdict
from datetime import datetime, timezone

import requests

try:
    import pypdf
    HAVE_PYPDF = True
except ImportError:
    HAVE_PYPDF = False

GRAPHQL_URL = "https://graphql.earthdata.nasa.gov/api"
SESSION = requests.Session()
SESSION.headers["User-Agent"] = "metadata-quality-skills/collections_jupyter"

os.makedirs(OUTPUT_DIR, exist_ok=True)
DOC_CACHE_DIR = os.path.join(OUTPUT_DIR, "doc_cache")
os.makedirs(DOC_CACHE_DIR, exist_ok=True)

VALID_ASSOCIATION_TYPES = {"SCIENCE ASSOCIATED", "DEPENDENT", "INPUT", "PARENT",
                           "CHILD", "RELATED", "LARGER CITATION WORKS"}
UPSTREAM_TYPES = {"INPUT", "PARENT"}      # related collection feeds the target
DOWNSTREAM_TYPES = {"CHILD"}              # related collection is produced from the target


def graphql(query, variables=None, retries=3):
    """POST a query to CMR GraphQL with simple retry/back-off."""
    last = None
    for attempt in range(retries):
        try:
            r = SESSION.post(GRAPHQL_URL, json={"query": query, "variables": variables or {}},
                             timeout=HTTP_TIMEOUT)
            r.raise_for_status()
            body = r.json()
            if body.get("errors"):
                raise RuntimeError(body["errors"][0].get("message"))
            return body["data"]
        except Exception as exc:  # network errors, 5xx, GraphQL errors
            last = exc
            time.sleep(2 * (attempt + 1))
    raise RuntimeError(f"GraphQL failed after {retries} attempts: {last}")


print(f"pypdf available: {HAVE_PYPDF}")
print(f"Output directory: {os.path.abspath(OUTPUT_DIR)}")

pypdf available: True
Output directory: C:\Users\lchen25\Documents\Esdis-Project\metadata-quality-skills\collections_jupyter\output


## Load the input file and profile it

Each record is one collection (`concept_id`, `target_collection`) with the associations that need repair under
`lineage_data.parent_collections`. `association_index` points at the position of that association in the live
`MetadataAssociations` array, so fixes can be merged back without touching the other entries.
`description_override` is a status note written by `find_schema_issues.py` — it is never copied into `Description`.

In [3]:
with open(INPUT_FILE, encoding="utf-8") as fh:
    records = json.load(fh)
if LIMIT_RECORDS:
    records = records[:LIMIT_RECORDS]

all_assocs = [a for r in records for a in r["lineage_data"]["parent_collections"]]
print(f"Collections: {len(records)}   associations to repair: {len(all_assocs)}")
print("Providers:   ", dict(Counter(r["concept_id"].split("-", 1)[1] for r in records).most_common()))
print("Flags:       ", dict(Counter(f for a in all_assocs for f in a.get("flags", [])).most_common()))
print("Types:       ", dict(Counter(a.get("original_type") for a in all_assocs).most_common()))
print("No atbd_url: ", sum(1 for r in records if not r.get("atbd_url")))
print("Distinct related identifiers:", len({a["original_id"] for a in all_assocs}))

Collections: 54   associations to repair: 139
Providers:    {'ASF': 35, 'LANCEMODIS': 14, 'LPCLOUD': 3, 'LPDAAC_ECS': 2}
Flags:        {'missing_version': 139, 'version_in_entryId': 3, 'embedded_version_not_in_cmr': 2}
Types:        {'RELATED': 72, 'SCIENCE ASSOCIATED': 28, 'INPUT': 23, 'DEPENDENT': 16}
No atbd_url:  10
Distinct related identifiers: 74


## Step 1 — CMR lookups and ATBD URL discovery (skill Step 1)

* `fetch_details` pulls the live record (related URLs, processing level, current `MetadataAssociations`) for every target.
* `lookup_short_name` is a case-insensitive exact ShortName lookup; `search_pattern` is a wildcard search; both are cached.
* `extract_atbd_urls_from_metadata` applies the skill's priority order:
  1 = `ALGORITHM THEORETICAL BASIS DOCUMENT (ATBD)` subtype, 2 = `VIEW RELATED INFORMATION` with ATBD in the subtype,
  3 = a `.pdf` URL containing "ATBD", 4 = user guide / product specification (fallback, as in `find_schema_issues.py`).

In [4]:
COLLECTION_FIELDS = "conceptId shortName versionId provider title processingLevel"
DETAIL_FIELDS = COLLECTION_FIELDS + " relatedUrls metadataAssociations"
COLLECTIONS_QUERY = ("query($params: CollectionsInput){ collections(params:$params){ count cursor items{ %s } } }")

_lookup_cache, _pattern_cache, _provider_cache = {}, {}, {}


def _level_id(item):
    pl = item.get("processingLevel")
    return (pl or {}).get("id") if isinstance(pl, dict) else pl


def _slim(item):
    return {"concept_id": item["conceptId"], "short_name": item["shortName"], "version": item["versionId"],
            "provider": item["provider"], "title": item.get("title") or "", "level": _level_id(item)}


def fetch_details(concept_ids, batch=50):
    """Live records keyed by concept ID. A failing batch is split in half until the bad ID is isolated."""
    out = {}

    def run(ids):
        try:
            data = graphql(COLLECTIONS_QUERY % DETAIL_FIELDS,
                           {"params": {"conceptId": ids, "limit": len(ids)}})
            for it in data["collections"]["items"]:
                out[it["conceptId"]] = it
        except RuntimeError as exc:
            if len(ids) == 1:
                print(f"  could not fetch {ids[0]}: {exc}")
            else:
                run(ids[: len(ids) // 2]); run(ids[len(ids) // 2:])

    for i in range(0, len(concept_ids), batch):
        run(concept_ids[i:i + batch])
    return out


def lookup_short_name(short_name):
    """All CMR collections whose ShortName equals `short_name` (case-insensitive). None = CMR unavailable."""
    key = short_name.lower()
    if key not in _lookup_cache:
        try:
            data = graphql(COLLECTIONS_QUERY % COLLECTION_FIELDS,
                           {"params": {"shortName": short_name, "limit": 100,
                                       "options": {"shortName": {"ignoreCase": True}}}})
            _lookup_cache[key] = [_slim(i) for i in data["collections"]["items"]]
        except RuntimeError:
            return None
    return _lookup_cache[key]


def search_pattern(pattern, provider=None, limit=200):
    """Wildcard ShortName search (`*` / `?`), optionally restricted to one provider."""
    key = (pattern.lower(), provider)
    if key not in _pattern_cache:
        params = {"shortName": pattern, "limit": limit,
                  "options": {"shortName": {"pattern": True, "ignoreCase": True}}}
        if provider:
            params["provider"] = provider
        try:
            data = graphql(COLLECTIONS_QUERY % COLLECTION_FIELDS, {"params": params})
            _pattern_cache[key] = [_slim(i) for i in data["collections"]["items"]]
        except RuntimeError:
            _pattern_cache[key] = []
    return _pattern_cache[key]


def provider_collections(provider):
    """Every collection of a provider (cursor paging) — used for typo and title matching."""
    if provider not in _provider_cache:
        items, cursor = [], None
        while True:
            params = {"provider": provider, "limit": 1000}
            if cursor:
                params["cursor"] = cursor
            try:
                page = graphql(COLLECTIONS_QUERY % COLLECTION_FIELDS, {"params": params})["collections"]
            except RuntimeError:
                break
            items += [_slim(i) for i in page["items"]]
            cursor = page.get("cursor")
            if not page["items"] or not cursor:
                break
        _provider_cache[provider] = items
    return _provider_cache[provider]


def extract_atbd_urls_from_metadata(related_urls, fallback_url=None):
    """Skill Step 1: candidate lineage documents ordered by priority (1 = best)."""
    found = {}
    for u in related_urls or []:
        url = (u.get("url") or u.get("URL") or "").strip()
        if not url:
            continue
        typ = (u.get("type") or u.get("Type") or "").upper()
        sub = (u.get("subtype") or u.get("Subtype") or "").upper()
        low = url.lower()
        if "ALGORITHM THEORETICAL BASIS DOCUMENT" in sub:
            prio = 1
        elif typ == "VIEW RELATED INFORMATION" and "ATBD" in sub:
            prio = 2
        elif "atbd" in low and low.endswith(".pdf"):
            prio = 3
        elif any(k in sub for k in ("USER'S GUIDE", "USER GUIDE", "PRODUCT SPECIFICATION",
                                    "DATA PRODUCT SPECIFICATION", "READ-ME")) or (
                typ == "VIEW RELATED INFORMATION" and low.endswith(".pdf")):
            prio = 4
        else:
            continue
        found[url] = min(prio, found.get(url, 9))
    if fallback_url and fallback_url not in found:
        found[fallback_url] = 3 if "atbd" in fallback_url.lower() else 4
    return sorted(({"url": k, "priority": v} for k, v in found.items()), key=lambda d: d["priority"])

## Step 2 — Fetch and validate the lineage documents (skill Step 2)

`fetch_document` downloads a PDF or HTML document once, caches the extracted text in `output/doc_cache/`,
and returns it as pages (`pypdf`) or as one block (fallback extractor). Documents larger than `MAX_DOC_BYTES`
are skipped. `validate_atbd_for_lineage` applies the skill's five criteria (at least four must pass). The skill's
hard-coded product prefixes (`MOD`, `VNP`, …) are generalized to "names the related collection or another
product-style identifier", so MISR, CERES, CALIPSO and Sentinel-1 documents are judged the same way.

In [5]:
PRODUCT_TOKEN = re.compile(r"\b[A-Z][A-Z0-9]{1,}[_-]?[A-Z0-9_.-]*\d[A-Za-z0-9_.-]*")
_doc_memory = {}


def _pdf_pages_stdlib(raw):
    """Approximate PDF text: inflate content streams and read Tj/TJ strings (no page numbers)."""
    chunks = []
    for m in re.finditer(rb"stream\r?\n(.*?)\r?\nendstream", raw, re.S):
        try:
            stream = zlib.decompress(m.group(1))
        except Exception:
            continue
        for block in re.findall(rb"BT(.*?)ET", stream, re.S):
            parts = re.findall(rb"\((.*?)(?<!\\)\)", block, re.S)
            if parts:
                chunks.append(re.sub(r"\\([()\\])", r"\1", b"".join(parts).decode("latin-1")))
    return [{"page": None, "text": "\n".join(chunks)}]


def _pdf_pages(raw):
    if HAVE_PYPDF:
        try:
            reader = pypdf.PdfReader(io.BytesIO(raw))
            return [{"page": i + 1, "text": p.extract_text() or ""} for i, p in enumerate(reader.pages)], True
        except Exception:
            pass
    return _pdf_pages_stdlib(raw), False


def _html_text(body):
    body = re.sub(r"(?is)<(script|style).*?</\1>", " ", body)
    body = re.sub(r"(?i)<br\s*/?>|</p>|</h\d>|</li>|</tr>", "\n", body)
    return htmllib.unescape(re.sub(r"<[^>]+>", " ", body))


def fetch_document(url):
    """{'url', 'ok', 'exact', 'pages': [{'page', 'text'}], 'error'} — cached on disk (text only)."""
    if url in _doc_memory:
        return _doc_memory[url]
    cache_file = os.path.join(DOC_CACHE_DIR, hashlib.sha1(url.encode()).hexdigest() + ".json")
    if os.path.exists(cache_file):
        with open(cache_file, encoding="utf-8") as fh:
            cached = json.load(fh)
        if cached.get("exact") or not HAVE_PYPDF:      # re-extract fallback text once pypdf is installed
            _doc_memory[url] = cached
            return cached

    doc = {"url": url, "ok": False, "exact": False, "pages": [], "error": None}
    if not FETCH_DOCUMENTS:
        doc["error"] = "FETCH_DOCUMENTS is False"
        _doc_memory[url] = doc
        return doc
    try:
        with SESSION.get(url, timeout=HTTP_TIMEOUT, stream=True, allow_redirects=True) as r:
            r.raise_for_status()
            raw = b""
            for chunk in r.iter_content(1 << 16):
                raw += chunk
                if len(raw) > MAX_DOC_BYTES:
                    raise ValueError(f"document larger than {MAX_DOC_BYTES // 1_000_000} MB")
        if raw[:5] == b"%PDF-":
            doc["pages"], doc["exact"] = _pdf_pages(raw)
        else:
            doc["pages"], doc["exact"] = [{"page": None, "text": _html_text(raw.decode("utf-8", "replace"))}], True
        doc["ok"] = any(p["text"].strip() for p in doc["pages"])
        if not doc["ok"]:
            doc["error"] = "no extractable text"
    except Exception as exc:
        doc["error"] = f"{type(exc).__name__}: {exc}"[:300]
    if doc["ok"]:
        with open(cache_file, "w", encoding="utf-8") as fh:
            json.dump(doc, fh)
    _doc_memory[url] = doc
    return doc


def doc_text(doc):
    return "\n".join(p["text"] for p in doc["pages"])


def validate_atbd_for_lineage(doc, related_names=()):
    """Skill Step 2 validation: five criteria, valid when at least four pass."""
    text = doc_text(doc) if doc.get("ok") else ""
    low = text.lower()
    squashed = re.sub(r"\s+", "", text).upper()
    checks = {
        "document_accessible": bool(text.strip()),
        "contains_algorithm_section": "algorithm" in low,
        "mentions_input_data": any(t in low for t in ("input", "derived from", "uses")),
        "mentions_parent_collection": any(re.sub(r"\s+", "", n).upper() in squashed for n in related_names if n)
                                      or bool(PRODUCT_TOKEN.search(text)),
        "contains_processing_algorithm": any(t in low for t in ("algorithm", "equation", "calculation", "retrieval")),
    }
    return sum(checks.values()) >= 4, checks

## Steps 3–4 — Lineage statements and document sections (skill Steps 3–4)

`extract_lineage_statements` finds the sentences that name the related collection and classifies them with the
skill's keyword priorities (P1 *derived from / aggregated from / composite of*, P2 *input / uses / based on*,
P3 *from / output of*). Each hit keeps the quote, page and nearest numbered section heading (Step 4).
Matching ignores spacing so it still works on the fallback extractor's broken words ("u ser g uide").

`documented_versions` collects *Collection N / Version N / Edition N / Release N / VNNN* statements in sentences
that name the product or its family — the ATBD evidence used to choose a Version.

In [6]:
LINEAGE_KEYWORDS = [
    (1, "derived_from", re.compile(r"derived\s+from|aggregated\s+from|composite\s+of|generated\s+from|produced\s+from", re.I)),
    (2, "input_to", re.compile(r"\binputs?\b|\buses\b|\bused\b|based\s+on|ancillary", re.I)),
    (3, "related", re.compile(r"\bfrom\b|output\s+of", re.I)),
]
HEADING = re.compile(r"(?m)^\s*(\d+(?:\.\d+){0,3})\.?\s+([A-Z][A-Za-z][^\n]{2,80})$")
VERSION_STATEMENT = re.compile(
    r"\b(?:Collection|Version|Edition|Release)\s*[-:]?\s*(\d+(?:\.\d+)?[A-Za-z]?)\b|\bV(\d{3})\b|\bv(\d+(?:\.\d+)?)\b")


def _sentences(text):
    for s in re.split(r"(?<=[.!?;])\s+|\n{2,}", text):
        s = re.sub(r"\s+", " ", s).strip()
        if 15 <= len(s) <= 1500:
            yield s


def doc_sentences(doc):
    """[(page, sentence)] — computed once per document."""
    if "_sentences" not in doc:
        doc["_sentences"] = [(p["page"], s) for p in doc["pages"] for s in _sentences(p["text"])]
        doc["_squashed"] = re.sub(r"\s+", "", doc_text(doc)).upper()
    return doc["_sentences"]


def _names_in(sentence, name):
    """Word-boundary match that tolerates broken spacing inside the name."""
    if not name:
        return False
    if re.search(rf"(?<![A-Za-z0-9]){re.escape(name)}(?![A-Za-z0-9])", sentence, re.I):
        return True
    return len(name) >= 6 and re.sub(r"\s+", "", name).upper() in re.sub(r"\s+", "", sentence).upper()


def section_for(page_text, quote):
    """Nearest numbered heading before the quote on its page (skill Step 4)."""
    heads = [(m.start(), f"{m.group(1)} {m.group(2).strip()}") for m in HEADING.finditer(page_text)]
    words = quote.split()[:6]
    hit = re.search(r"\s+".join(map(re.escape, words)), page_text) if heads and words else None
    if not hit:
        return None
    before = [h for p, h in heads if p <= hit.start()]
    return before[-1] if before else None


def extract_lineage_statements(doc, related_name, target_name, max_hits=5):
    """[{priority, relationship, quote, page, section, names_target}] best first."""
    hits = []
    if not doc.get("ok") or not related_name:
        return hits
    doc_sentences(doc)
    if re.sub(r"\s+", "", related_name).upper() not in doc["_squashed"]:
        return hits                                   # fast exit: name never appears
    page_text = {p["page"]: p["text"] for p in doc["pages"]}
    for page, s in doc["_sentences"]:
        if not _names_in(s, related_name):
            continue
        prio, rel = 9, "mentioned"
        for p, r, rx in LINEAGE_KEYWORDS:
            if rx.search(s):
                prio, rel = p, r
                break
        hits.append({"priority": prio, "relationship": rel, "quote": s[:600], "page": page,
                     "section": section_for(page_text.get(page, ""), s), "names_target": _names_in(s, target_name)})
    hits.sort(key=lambda h: (h["priority"], not h["names_target"]))
    return hits[:max_hits]


def product_family(short_name):
    """Leading letters+digits of a ShortName: MOD09CMA -> MOD09, MI1B2E -> MI1, CER_SSF1DEG -> CER_SSF1."""
    m = re.match(r"[A-Za-z_]+\d+", short_name or "")
    return m.group(0) if m else (short_name or "")


def documented_versions(doc, short_name):
    """[(version, quote, page)] stated in sentences naming the product (or its family)."""
    out = []
    if not doc.get("ok"):
        return out
    names = [n for n in {short_name, product_family(short_name)} if len(n) >= 4]
    doc_sentences(doc)
    names = [n for n in names if re.sub(r"\s+", "", n).upper() in doc["_squashed"]]
    for page, s in doc["_sentences"] if names else []:
        if not any(_names_in(s, n) for n in names):
            continue
        for m in VERSION_STATEMENT.finditer(s):
            v = next(g for g in m.groups() if g)
            out.append((v, s[:500], page))
    return out

## Step 6 — Validate against CMR and choose the Version (skill Step 6)

`validate_against_cmr` checks that the related collection exists and that the processing level progresses in the
direction the association `Type` implies (an `INPUT`/`PARENT` is at or below the target's level, a `CHILD` at or above).
Other types (`RELATED`, `SCIENCE ASSOCIATED`, `DEPENDENT`) are not level-checked. Version alignment is recorded
for the reviewer but is not a pass/fail test, because inputs from other instruments use unrelated version schemes.

`choose_version` picks one CMR version for the related ShortName, in this order:

| Basis | Rule |
|---|---|
| `original_verified` | the version the provider wrote (or embedded in the identifier) exists in CMR |
| `unique_in_cmr` | only one version exists (the target itself excluded by concept ID) |
| `documented_in_atbd` | the lineage document states exactly one of the CMR versions |
| `matches_target` | one candidate has the same version as the target |
| `ranked_latest` | none of the above: same provider > non-NRT > latest — flagged `multiple_versions_in_cmr` |

If the only CMR match is the target itself, the result is `self_reference` and a separate resolver takes over.

In [7]:
def norm_version(v):
    s = str(v or "").strip().lower()
    s = re.sub(r"^(version|edition|ed|v)\s*", "", s)
    return s.lstrip("0") or "0"


def same_version(a, b):
    """'2' == 'v002' == 'Version 2', '6.1' == '061'."""
    if a is None or b is None:
        return False
    na, nb = norm_version(a), norm_version(b)
    return na == nb or na.replace(".", "").lstrip("0") == nb.replace(".", "").lstrip("0")


def version_sort_key(v):
    return [(0, int(t), "") if t.isdigit() else (1, 0, t.lower()) for t in re.findall(r"\d+|[A-Za-z]+", str(v or ""))]


def is_nrt(c):
    return ("NRT" in str(c.get("version", "")).upper() or str(c.get("short_name", "")).upper().endswith("_NRT")
            or str(c.get("provider", "")).upper().startswith("LANCE"))


def describe(c):
    return f"{c['short_name']} v{c['version']} ({c['concept_id']})"


def level_rank(level):
    """'1A' -> 1.01, '1B' -> 1.02, '1T' -> 1.2, '2' -> 2, 'L3' -> 3; 'NA' / 'Not provided' -> None."""
    m = re.search(r"(\d)\s*([A-Za-z])?", str(level or ""))
    if not m:
        return None
    letter = (m.group(2) or "").upper()
    return int(m.group(1)) + ((ord(letter) - ord("A") + 1) / 100 if letter else 0)


def validate_against_cmr(related, target_ctx, assoc_type):
    """Skill Step 6 checks for one resolved association."""
    rel_rank, tgt_rank = level_rank(related.get("level")), level_rank(target_ctx.get("level"))
    if assoc_type in UPSTREAM_TYPES or assoc_type in DOWNSTREAM_TYPES:
        if rel_rank is None or tgt_rank is None:
            progression = "unknown"
        elif assoc_type in UPSTREAM_TYPES:
            progression = "ok" if rel_rank <= tgt_rank else "invalid"
        else:
            progression = "ok" if rel_rank >= tgt_rank else "invalid"
    else:
        progression = "not_applicable"
    return {"exists": True, "related_concept_id": related["concept_id"], "related_level": related.get("level"),
            "target_level": target_ctx.get("level"), "level_progression": progression,
            "version_alignment": "same" if same_version(related["version"], target_ctx["version"]) else "different"}


def _rank_candidates(cands, target_ctx):
    ranked = sorted(cands, key=lambda c: version_sort_key(c["version"]), reverse=True)
    ranked.sort(key=lambda c: (c["provider"] != target_ctx["provider"], is_nrt(c)))
    return ranked


def choose_version(candidates, target_ctx, original_version=None, doc_versions=()):
    """-> {'candidate', 'basis', 'flags', 'quote', 'candidates'}; candidate is None for self_reference."""
    others = [c for c in candidates if c["concept_id"] != target_ctx["concept_id"]]
    listing = [describe(c) for c in _rank_candidates(candidates, target_ctx)]
    if not others:
        return {"candidate": None, "basis": "self_reference", "flags": ["self_reference_only"],
                "quote": None, "candidates": listing}

    def doc_support(cands):
        hits = [(c, q, p) for v, q, p in doc_versions for c in cands if same_version(c["version"], v)]
        return hits

    if original_version:
        hit = [c for c in others if same_version(c["version"], original_version)]
        if hit:
            return {"candidate": _rank_candidates(hit, target_ctx)[0], "basis": "original_verified",
                    "flags": [], "quote": None, "candidates": listing}

    versions = {c["version"] for c in others}
    if len(versions) == 1:
        support = doc_support(others)
        return {"candidate": _rank_candidates(others, target_ctx)[0], "basis": "unique_in_cmr", "flags": [],
                "quote": support[0][1] if support else None, "candidates": listing}

    support = doc_support(others)
    if support and len({c["version"] for c, _, _ in support}) == 1:
        c, q, p = support[0]
        return {"candidate": c, "basis": "documented_in_atbd", "flags": [], "quote": q, "page": p,
                "candidates": listing}

    same = [c for c in others if same_version(c["version"], target_ctx["version"])]
    if same:
        return {"candidate": _rank_candidates(same, target_ctx)[0], "basis": "matches_target", "flags": [],
                "quote": None, "candidates": listing}

    return {"candidate": _rank_candidates(others, target_ctx)[0], "basis": "ranked_latest",
            "flags": ["multiple_versions_in_cmr"], "quote": None, "candidates": listing}

## Shared resolvers — self-references and typos

Used by both notebooks when the CMR lookup alone cannot settle an association:

* **Standard counterpart** — a LANCE NRT collection (`MOD09A1N`, `X_NRT`) whose association points to itself is
  linked to its standard product (`MOD09A1`) when that exists in CMR.
* **Documented companion** — when there is no standard counterpart, read the target's documents and the
  documents of its product family's standard collections, and link the product most often named in the same
  sentence as the target (unique, in CMR, not NRT). This is how `MOD09CMA → MOD09CMG` is found.
* **Typo correction** — a unique ShortName within the target's provider with difflib similarity ≥ `FUZZY_CUTOFF`.

In [8]:
DOC_SUBTYPE_PRIORITY = ("ALGORITHM THEORETICAL BASIS DOCUMENT (ATBD)", "USER'S GUIDE")
_family_docs_cache = {}


def standard_counterpart(target_ctx):
    """Non-NRT collection whose ShortName is the NRT ShortName without `_NRT` or a trailing `N`."""
    name = target_ctx["short_name"].strip()
    for pattern in (r"_NRT$", r"(?<=\d)N$", r"(?<=[A-Z]\d[A-Z]\d)N$"):
        stripped = re.sub(pattern, "", name, flags=re.I)
        if stripped == name:
            continue
        standard = [c for c in lookup_short_name(stripped) or [] if not is_nrt(c)]
        if standard:
            return sorted(standard, key=lambda c: version_sort_key(c["version"]), reverse=True)[0]
    return None


def family_documentation_urls(short_name):
    """[(url, subtype, source)] ATBDs / user guides linked from the family's standard (non-NRT) collections."""
    family = product_family(short_name)
    if family in _family_docs_cache:
        return _family_docs_cache[family]
    docs, seen = [], set()
    try:
        data = graphql(COLLECTIONS_QUERY % (COLLECTION_FIELDS + " relatedUrls"),
                       {"params": {"shortName": f"{family}*", "limit": 200,
                                   "options": {"shortName": {"pattern": True, "ignoreCase": True}}}})
        items = data["collections"]["items"]
    except RuntimeError:
        items = []
    for it in items:
        c = _slim(it)
        if is_nrt(c):
            continue
        for u in it.get("relatedUrls") or []:
            sub = str(u.get("subtype") or "").upper()
            if u.get("url") and sub in DOC_SUBTYPE_PRIORITY and u["url"] not in seen:
                seen.add(u["url"])
                docs.append((u["url"], sub, describe(c)))
    docs.sort(key=lambda d: DOC_SUBTYPE_PRIORITY.index(d[1]))
    _family_docs_cache[family] = docs
    return docs


def documented_companion(target_ctx, doc_urls, max_docs=4):
    """Most-cited same-family product named alongside the target in its (or its family's) documents."""
    target = target_ctx["short_name"]
    family = product_family(target).upper()
    sources = [(u, "target document", target) for u in doc_urls] + family_documentation_urls(target)
    for url, subtype, source in sources[:max_docs]:
        doc = fetch_document(url)
        if not doc["ok"]:
            continue
        counts = defaultdict(list)
        for page, s in doc_sentences(doc):
            if not _names_in(s, target):
                continue
            for tok in set(re.findall(r"\b[A-Z]{2,}\d+[A-Za-z0-9_]*", s)):
                if tok != target and tok.upper().startswith(family):
                    counts[tok].append((s, page))
        ranked = []
        for name, sents in counts.items():
            standard = [c for c in lookup_short_name(name) or [] if not is_nrt(c)]
            if standard:
                ranked.append((len(sents), name, sents, standard))
        ranked.sort(key=lambda r: r[0], reverse=True)
        if not ranked or (len(ranked) > 1 and ranked[0][0] == ranked[1][0]):
            continue
        n, name, sents, standard = ranked[0]
        best = sorted(standard, key=lambda c: version_sort_key(c["version"]), reverse=True)[0]
        quote, page = max(sents, key=lambda sp: sp[0].count(target) + sp[0].count(name))
        return {"candidate": best, "url": url, "quote": quote[:600], "page": page, "exact": doc["exact"],
                "evidence": (f"{subtype} {url} (linked from {source}) names {name} in the same sentence as "
                             f"{target} {n} time(s), more than any other product in CMR")}
    return None


def typo_match(identifier, provider):
    """(ShortName, similarity) of the unique close match inside the provider, else None.
    The target itself is not excluded: a typo of the target's own name becomes a self-reference."""
    by_lower = {c["short_name"].lower(): c["short_name"] for c in provider_collections(provider)}
    close = difflib.get_close_matches(identifier.lower(), sorted(by_lower), n=2, cutoff=FUZZY_CUTOFF)
    if len(close) != 1:
        return None
    return by_lower[close[0]], round(difflib.SequenceMatcher(None, identifier.lower(), close[0]).ratio(), 3)

## Confidence (adapted from the CSV Export skill rubric)

The provider already asserted each association, so confidence here describes **how sure we are of the repaired
ShortName + Version**, using three criteria — identifier match, version basis, CMR validation:

| Level | Rule |
|---|---|
| **HIGH** | strong identifier (exact / case-normalized / verified embedded version) **and** strong version basis (`original_verified`, `unique_in_cmr`, `documented_in_atbd`, `matches_target`) **and** CMR checks pass |
| **LOW** | unresolved, self-reference, not in CMR, invalid level progression, or a weak identifier (prefix family) with no document evidence |
| **MEDIUM** | everything else — e.g. typo correction, title match, versioned-family match, documented companion, `ranked_latest` |

`confidence_reason` spells out which criterion held each row back, so a reviewer can see why it is not HIGH.

In [9]:
STRONG_ID = {"exact", "case_normalized", "embedded_verified"}
MODERATE_ID = {"typo_corrected", "title_match", "versioned_family", "standard_counterpart",
               "documented_companion", "embedded_split", "extracted_from_text"}
WEAK_ID = {"prefix_family"}
STRONG_VERSION = {"original_verified", "unique_in_cmr", "documented_in_atbd", "matches_target", "embedded_verified"}


def assess_confidence(status, id_basis, version_basis, cmr_check, has_doc_evidence):
    if status != "resolved":
        return "LOW", f"not resolved ({status})"
    reasons = []
    if cmr_check and cmr_check.get("level_progression") == "invalid":
        return "LOW", (f"processing level progression invalid "
                       f"(related {cmr_check['related_level']} vs target {cmr_check['target_level']})")
    if id_basis in WEAK_ID and not has_doc_evidence:
        return "LOW", f"weak identifier match ({id_basis}) with no document evidence"
    id_ok = id_basis in STRONG_ID
    ver_ok = version_basis in STRONG_VERSION
    if not id_ok:
        reasons.append(f"identifier: {id_basis}")
    if not ver_ok:
        reasons.append(f"version: {version_basis}")
    if id_ok and ver_ok:
        return "HIGH", f"identifier {id_basis}; version {version_basis}; CMR {cmr_check.get('level_progression')}" \
                       + ("; document evidence" if has_doc_evidence else "")
    return "MEDIUM", "; ".join(reasons) + ("; document evidence" if has_doc_evidence else "")

## Skill Steps 5 & 8 — Build and validate the UMM-C `MetadataAssociations` entries

Under the new rules the related collection is identified by **ShortName** (not EntryId) and **Version is mandatory**.
`ASSOCIATION_ID_FIELD` controls the output key (`"ShortName"`, or `"EntryId"` for UMM-C ≤ 1.18.x).

* The original `Type` is kept (it is a valid enum value in every input record).
* The original `Description` is kept. If there is none, one is generated **only** when the document gives a lineage
  quote; status notes (`description_override`) never go into `Description`.
* `validate_association` checks: identifier and Version present, length limits (ShortName 1–85, EntryId 1–80,
  Version 1–80, Description 1–4000), the Type enum, and no extra fields.

In [10]:
ID_MAX_LEN = 85 if ASSOCIATION_ID_FIELD == "ShortName" else 80
REL_PHRASE = {"INPUT": "is an input to", "PARENT": "is a parent (source) collection of",
              "CHILD": "is derived from", "DEPENDENT": "depends on", "RELATED": "is related to",
              "SCIENCE ASSOCIATED": "is scientifically associated with", "LARGER CITATION WORKS": "is cited with"}


def generate_description(related, target, assoc_type, statement):
    if not statement:
        return None
    if assoc_type == "CHILD":
        text = f"{related} is derived from {target}."
    else:
        text = f"{related} {REL_PHRASE.get(assoc_type, 'is related to')} {target}."
    where = ", ".join(x for x in [statement.get("section") and f"section {statement['section']}",
                                  statement.get("page") and f"page {statement['page']}"] if x)
    text += f" Source ({statement['doc']}{', ' + where if where else ''}): \"{statement['quote']}\""
    return text[:3997] + "..." if len(text) > 4000 else text


def build_association(assoc_type, short_name, version, description):
    a = {"Type": assoc_type}
    if description:
        a["Description"] = description[:4000]
    a[ASSOCIATION_ID_FIELD] = short_name
    if version:
        a["Version"] = str(version)
    return a


def validate_association(a):
    errors = []
    ident = a.get(ASSOCIATION_ID_FIELD)
    if not ident:
        errors.append(f"{ASSOCIATION_ID_FIELD} is required")
    elif len(ident) > ID_MAX_LEN:
        errors.append(f"{ASSOCIATION_ID_FIELD} longer than {ID_MAX_LEN}")
    ver = a.get("Version")
    if not ver or not str(ver).strip():
        errors.append("Version is required")
    elif len(str(ver)) > 80:
        errors.append("Version longer than 80")
    if a.get("Type") not in VALID_ASSOCIATION_TYPES:
        errors.append(f"Type {a.get('Type')!r} not in enum")
    if "Description" in a and not (1 <= len(a["Description"]) <= 4000):
        errors.append("Description must be 1-4000 characters")
    extra = set(a) - {"Type", "Description", ASSOCIATION_ID_FIELD, "Version"}
    if extra:
        errors.append(f"unexpected fields {sorted(extra)}")
    return errors


def live_to_umm(a):
    """GraphQL camelCase association -> UMM-C keys (identifier moved to ASSOCIATION_ID_FIELD)."""
    out = {"Type": a.get("type")}
    if a.get("description"):
        out["Description"] = a["description"]
    out[ASSOCIATION_ID_FIELD] = a.get("shortName") or a.get("entryId")
    if a.get("version"):
        out["Version"] = str(a["version"])
    return out


print(validate_association(build_association("INPUT", "MOD11A1", "061", "test")))
print(validate_association(build_association("INPUT", "MOD11A1", None, None)))

[]
['Version is required']


## Identifier resolution for missing-version associations

`find_schema_issues.py` already pre-filled the version for the three `version_in_entryId` cases it could verify.
Those are re-checked here rather than trusted. The `embedded_version_not_in_cmr` cases (`VNP46A1_1`) keep the
base ShortName and let `choose_version` decide from CMR and the user guide (*"Collection 2 … VNP46"*).

In [11]:
HIDDEN_VERSION_PATTERN = re.compile(r"_(v?\d[\w.]*)$", re.IGNORECASE)   # same pattern as find_schema_issues.py


def _resolved(cands, basis, hint=None, flags=(), evidence=()):
    if not cands:
        return {"status": "not_in_cmr", "flags": list(flags) + ["identifier_not_in_cmr"], "evidence": list(evidence),
                "recommendation": "Matched name could not be re-read from CMR; review manually."}
    return {"status": "resolved", "short_name": cands[0]["short_name"], "candidates": cands, "id_basis": basis,
            "version_hint": hint, "flags": list(flags), "evidence": list(evidence)}


def resolve_identifier(a, target_ctx, docs):
    oid = (a.get("original_id") or "").strip()
    if not oid:
        return {"status": "missing_identifier", "recommendation": "Association has no identifier; remove it."}

    cands = lookup_short_name(oid)
    if cands is None:
        return {"status": "cmr_unavailable", "recommendation": "CMR could not be reached; re-run."}
    if cands:
        exact = any(c["short_name"] == oid for c in cands)
        return _resolved(cands, "exact" if exact else "case_normalized", a.get("original_version"),
                         evidence=[] if exact else [f"CMR spelling is {cands[0]['short_name']}"])

    m = HIDDEN_VERSION_PATTERN.search(oid)
    if m:
        base, ver = oid[:m.start()], m.group(1)
        base_cands = lookup_short_name(base) or []
        if base_cands:
            if any(same_version(c["version"], ver) for c in base_cands):
                return _resolved(base_cands, "embedded_verified", ver,
                                 evidence=[f"'{oid}' split into ShortName {base} + Version {ver}; both exist in CMR"])
            return _resolved(base_cands, "embedded_split", None, flags=["embedded_version_not_in_cmr"],
                             evidence=[f"'{oid}' split into {base} + {ver}, but CMR has no {base} v{ver} "
                                       f"(has {', '.join(sorted({c['version'] for c in base_cands}))})"])

    typo = typo_match(oid, target_ctx["provider"])
    if typo:
        return _resolved(lookup_short_name(typo[0]) or [], "typo_corrected", a.get("original_version"),
                         flags=["identifier_typo_corrected"],
                         evidence=[f"'{oid}' is not in CMR; unique close {target_ctx['provider']} match "
                                   f"'{typo[0]}' (similarity {typo[1]})"])

    return {"status": "not_in_cmr", "short_name": oid, "flags": ["identifier_not_in_cmr"],
            "recommendation": f"'{oid}' is not a CMR ShortName and has no unique close match; "
                              f"ask the provider for the intended collection."}


demo_ctx = {"concept_id": "C0-TEST", "short_name": "TEST", "version": "2", "provider": "ASF", "level": "1"}
for oid in ["SENTINEL-1A_SLC", "VNP46A1_2", "VNP46A1_1", "SENIINEL-1C_SLC"]:
    r = resolve_identifier({"original_id": oid}, demo_ctx, [])
    print(f"{oid:<18} -> {r['status']:<10} {r.get('short_name')!s:<18} {r.get('id_basis')!s:<18} "
          f"versions={sorted({c['version'] for c in r.get('candidates', [])})}")

SENTINEL-1A_SLC    -> resolved   SENTINEL-1A_SLC    exact              versions=['1']


VNP46A1_2          -> resolved   VNP46A1            embedded_verified  versions=['2']


VNP46A1_1          -> resolved   VNP46A1            embedded_split     versions=['2']


SENIINEL-1C_SLC    -> resolved   SENTINEL-1C_SLC    typo_corrected     versions=['1']


## Per-association pipeline (skill Steps 1–6 for one association)

`repair_association` runs the full chain for one entry of `parent_collections`:

1. `resolve_identifier` (notebook-specific, defined above) → the CMR ShortName and its candidate versions.
2. `choose_version` with the version the provider wrote and the versions stated in the validated documents.
3. Self-reference fallback → standard counterpart, then documented companion.
4. `validate_against_cmr` → existence + processing-level progression.
5. `extract_lineage_statements` → best quote naming the related collection (also tried with the original identifier).
6. `assess_confidence`, then `build_association` + `validate_association`.

In [12]:
def best_statement(docs, names, target_name):
    """Best lineage statement across the validated documents for any of `names`."""
    best = None
    for d in docs:
        if not d["valid"]:
            continue
        for n in names:
            for h in extract_lineage_statements(d["doc"], n, target_name, max_hits=1):
                h = dict(h, url=d["url"], doc=d["url"].rstrip("/").rsplit("/", 1)[-1], exact=d["doc"]["exact"],
                         matched_name=n)
                if best is None or (h["priority"], not h["names_target"]) < (best["priority"], not best["names_target"]):
                    best = h
    return best


def repair_association(a, target_ctx, docs):
    assoc_type = a.get("original_type") or a.get("relationship_type") or "RELATED"
    fix = {"concept_id": target_ctx["concept_id"], "collection": target_ctx["short_name"],
           "collection_version": target_ctx["version"], "association_index": a.get("association_index"),
           "type": assoc_type, "original_id": a.get("original_id"), "original_version": a.get("original_version"),
           "input_flags": a.get("flags", []), "input_available_versions": a.get("available_versions", [])}

    ident = resolve_identifier(a, target_ctx, docs)
    status, id_basis = ident["status"], ident.get("id_basis")
    flags = list(ident.get("flags", []))
    evidence = list(ident.get("evidence", []))
    choice, candidate, quote_info = None, None, None

    if status == "resolved":
        doc_versions = [v for d in docs if d["valid"] for v in documented_versions(d["doc"], ident["short_name"])]
        choice = choose_version(ident["candidates"], target_ctx, ident.get("version_hint"), doc_versions)
        candidate = choice["candidate"]
        if candidate is None and is_nrt(target_ctx):           # NRT target pointing at itself
            counterpart = standard_counterpart(target_ctx)
            companion = None if counterpart else documented_companion(target_ctx, [d["url"] for d in docs])
            replacement = counterpart or (companion or {}).get("candidate")
            if replacement:
                id_basis = "standard_counterpart" if counterpart else "documented_companion"
                flags.append("self_reference_replaced")
                evidence.append(f"association pointed to the target itself; replaced by {id_basis} "
                                f"{describe(replacement)}" + (f" — {companion['evidence']}" if companion else ""))
                pool = [c for c in lookup_short_name(replacement["short_name"]) or [] if not is_nrt(c)]
                choice = choose_version(pool or [replacement], target_ctx)
                candidate = choice["candidate"]
                if companion:
                    quote_info = {"quote": companion["quote"], "page": companion["page"], "url": companion["url"],
                                  "doc": companion["url"].rsplit("/", 1)[-1], "exact": companion["exact"],
                                  "section": None, "priority": 9, "relationship": "documented_companion"}
            else:
                status = "self_reference"
        if candidate is None:
            status = "self_reference"
        flags += choice.get("flags", [])
        if choice.get("quote"):
            evidence.append(f"document states version: \"{choice['quote'][:300]}\"")

    cmr = validate_against_cmr(candidate, target_ctx, assoc_type) if candidate else None
    names = [n for n in {candidate["short_name"] if candidate else None, a.get("original_id")} if n]
    statement = best_statement(docs, names, target_ctx["short_name"]) or quote_info
    if status in ("not_in_cmr",) and statement:
        status = "documented_not_in_cmr"
    has_doc = bool(statement) or bool(choice and choice.get("quote"))
    confidence, reason = assess_confidence(status, id_basis, choice["basis"] if choice else None, cmr, has_doc)

    new_assoc, errors = None, []
    if status == "resolved" and candidate:
        description = (a.get("original_description") or "").strip() or None
        if not description and statement and statement.get("exact"):
            description = generate_description(candidate["short_name"], target_ctx["short_name"], assoc_type, statement)
        new_assoc = build_association(assoc_type, candidate["short_name"], candidate["version"], description)
        errors = validate_association(new_assoc)
        if errors:
            status, confidence, reason = "invalid_output", "LOW", "; ".join(errors)

    fix.update({
        "status": status, "short_name": candidate["short_name"] if candidate else ident.get("short_name"),
        "version": candidate["version"] if candidate else None,
        "related_concept_id": candidate["concept_id"] if candidate else None,
        "id_basis": id_basis, "version_basis": choice["basis"] if choice else None,
        "confidence": confidence, "confidence_reason": reason,
        "flags": sorted(set(flags)), "evidence": evidence,
        "candidates": choice["candidates"] if choice else ident.get("alternatives", []),
        "cmr_validation": cmr,
        "lineage_statement": statement and {k: statement.get(k) for k in
                                            ("relationship", "priority", "quote", "page", "section", "doc", "url",
                                             "exact", "matched_name")},
        "recommendation": ident.get("recommendation") if status != "resolved" else (
            None if confidence == "HIGH" else "Review before ingest: " + reason),
        "new_association": new_assoc, "validation_errors": errors,
    })
    if status == "self_reference":
        fix["recommendation"] = ("The only CMR match is the target itself and no standard counterpart or documented "
                                 "companion exists. Remove the self-association or ask the provider which "
                                 "collection it should reference.")
    if status == "documented_not_in_cmr":
        fix["recommendation"] = (f"'{a.get('original_id')}' is named in the lineage document but has no CMR "
                                 f"collection. Remove the association, or ask the provider to register the "
                                 f"collection so it can be referenced by ShortName + Version.")
    return fix

## Per-collection processing (Steps 1–2 for the target, then every association)

In [13]:
def target_context(rec, live):
    tc = rec["target_collection"]
    return {"concept_id": rec["concept_id"], "short_name": tc["short_name"], "version": str(tc.get("version") or ""),
            "provider": rec["concept_id"].split("-", 1)[1], "title": tc.get("entry_title") or "",
            "level": (_level_id(live) if live else None) or tc.get("processing_level_id")}


def collect_documents(rec, live, related_names):
    urls = extract_atbd_urls_from_metadata((live or {}).get("relatedUrls"), rec.get("atbd_url"))
    docs = []
    for u in urls[:MAX_DOCS_PER_COLLECTION]:
        doc = fetch_document(u["url"])
        valid, checks = validate_atbd_for_lineage(doc, related_names)
        docs.append({"url": u["url"], "priority": u["priority"], "doc": doc, "valid": valid, "checks": checks})
    return docs


def merge_into_live(live_assocs, fixes):
    """Apply fixes to the live MetadataAssociations array (by association_index, checked against original_id)."""
    pending = list(fixes)
    out, applied = [], []
    for i, a in enumerate(live_assocs or []):
        ident = (a.get("shortName") or a.get("entryId") or "").strip()
        fix = next((f for f in pending if f["association_index"] == i and f["original_id"].lower() == ident.lower()),
                   None) or next((f for f in pending if f["original_id"].lower() == ident.lower()
                                  and not (a.get("version") and not f["original_version"])), None)
        if fix is None:
            out.append(live_to_umm(a))
            continue
        pending.remove(fix)
        applied.append(fix)
        if fix["new_association"]:
            out.append(fix["new_association"])
        elif not REMOVE_UNRESOLVED:
            out.append(live_to_umm(a))
    deduped, seen = [], set()
    for a in out:
        key = (str(a.get(ASSOCIATION_ID_FIELD)).lower(), a.get("Version"), a.get("Type"))
        if key not in seen:
            seen.add(key)
            deduped.append(a)
    return deduped, applied, pending

## Batch run

Fetches the live CMR record of every target, collects and validates its lineage documents, repairs each listed
association, and merges the fixes into the live `MetadataAssociations` array. Entries that were not in the input
are carried over unchanged (converted to `ASSOCIATION_ID_FIELD`); `remaining_invalid` counts the ones that still
break the new rules (e.g. no Version) — they are handled by the other notebook.

In [14]:
t0 = time.time()
live_by_id = fetch_details([r["concept_id"] for r in records]) if USE_LIVE_CMR else {}
print(f"Live CMR records fetched: {len(live_by_id)}/{len(records)}")

all_fixes, wrappers, doc_rows = [], [], {}
for n, rec in enumerate(records, 1):
    live = live_by_id.get(rec["concept_id"])
    target_ctx = target_context(rec, live)
    assocs = rec["lineage_data"]["parent_collections"]
    docs = collect_documents(rec, live, [a["original_id"] for a in assocs])
    for d in docs:
        doc_rows[d["url"]] = {"url": d["url"], "priority": d["priority"], "ok": d["doc"]["ok"],
                              "exact_text": d["doc"]["exact"], "valid_for_lineage": d["valid"],
                              "error": d["doc"]["error"], **d["checks"]}
    fixes = [repair_association(a, target_ctx, docs) for a in assocs]
    all_fixes += fixes

    if live is not None:
        merged, applied, unmatched = merge_into_live(live.get("metadataAssociations"), fixes)
        for f in unmatched:
            f["flags"].append("not_found_in_live_record")
    else:
        merged = [f["new_association"] for f in fixes if f["new_association"]]
    remaining_invalid = sum(1 for m in merged if validate_association(m))
    wrappers.append({
        "conceptId": rec["concept_id"], "collection_shortName": target_ctx["short_name"],
        "collection_version": target_ctx["version"], "collection_title": target_ctx["title"],
        "documents": [{"url": d["url"], "priority": d["priority"], "valid_for_lineage": d["valid"]} for d in docs],
        "associations_fixed": sum(1 for f in fixes if f["new_association"]),
        "associations_unresolved": sum(1 for f in fixes if not f["new_association"]),
        "confidence": dict(Counter(f["confidence"] for f in fixes)),
        "remaining_invalid": remaining_invalid,
        "source": "live CMR" if live is not None else "input file only",
        "umm_c_patch": {"ShortName": target_ctx["short_name"], "Version": target_ctx["version"],
                        "MetadataAssociations": merged},
    })
    if n % 25 == 0 or n == len(records):
        print(f"  {n}/{len(records)} collections processed ({time.time() - t0:.0f}s)")

Live CMR records fetched: 54/54


Ignoring wrong pointing object 6 0 (offset 0)


Ignoring wrong pointing object 9 0 (offset 0)


Ignoring wrong pointing object 17 0 (offset 0)


Ignoring wrong pointing object 20 0 (offset 0)


Ignoring wrong pointing object 22 0 (offset 0)


... (96 outputs left out)

Ignoring wrong pointing object 63 0 (offset 0)


Ignoring wrong pointing object 101 0 (offset 0)


Ignoring wrong pointing object 116 0 (offset 0)


  54/54 collections processed (194s)


## Save outputs

| File | Content |
|---|---|
| `{prefix}_fixes.json` | one entry per input association: before/after, id and version basis, confidence + reason, evidence, ATBD quote, CMR checks |
| `{prefix}_repaired_ummc.json` | per collection: the repaired `MetadataAssociations` array (`umm_c_patch`) — merge into the native UMM-C record before ingest |
| `{prefix}_review_report.csv` | flat review table, one row per association |
| `{prefix}_identifier_summary.csv` | grouped by original identifier — fix once, apply everywhere |
| `{prefix}_documents.csv` | every document fetched and its Step 2 validation |
| `{prefix}_lineage_report.md` | the skill's Step 8 report |

In [15]:
def out_path(name):
    return os.path.join(OUTPUT_DIR, f"{OUTPUT_PREFIX}_{name}")


with open(out_path("fixes.json"), "w", encoding="utf-8") as fh:
    json.dump(all_fixes, fh, indent=2)
with open(out_path("repaired_ummc.json"), "w", encoding="utf-8") as fh:
    json.dump(wrappers, fh, indent=2)

REVIEW_FIELDS = ["concept_id", "collection", "collection_version", "association_index", "type", "original_id",
                 "original_version", "status", "short_name", "version", "related_concept_id", "id_basis",
                 "version_basis", "confidence", "confidence_reason", "level_progression", "flags", "candidates",
                 "quote", "quote_doc", "quote_page", "recommendation"]
with open(out_path("review_report.csv"), "w", encoding="utf-8", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=REVIEW_FIELDS, extrasaction="ignore")
    w.writeheader()
    for f in all_fixes:
        st = f["lineage_statement"] or {}
        w.writerow({**f, "flags": ";".join(f["flags"]), "candidates": " | ".join(f["candidates"][:8]),
                    "level_progression": (f["cmr_validation"] or {}).get("level_progression"),
                    "quote": st.get("quote"), "quote_doc": st.get("doc"), "quote_page": st.get("page")})

groups = defaultdict(list)
for f in all_fixes:
    groups[f["original_id"]].append(f)
with open(out_path("identifier_summary.csv"), "w", encoding="utf-8", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["original_id", "occurrences", "collections", "statuses", "resolved_to", "confidence", "id_basis",
                "example_description", "recommendation"])
    for oid, fs in sorted(groups.items(), key=lambda kv: -len(kv[1])):
        w.writerow([oid, len(fs), len({f["concept_id"] for f in fs}),
                    ";".join(f"{k}={v}" for k, v in Counter(f["status"] for f in fs).items()),
                    ";".join(sorted({f"{f['short_name']} v{f['version']}" for f in fs if f["version"]})),
                    ";".join(f"{k}={v}" for k, v in Counter(f["confidence"] for f in fs).items()),
                    ";".join(sorted({str(f["id_basis"]) for f in fs})),
                    next((a.get("original_description") for r in records for a in r["lineage_data"]["parent_collections"]
                          if a["original_id"] == oid and a.get("original_description")), "") or "",
                    fs[0]["recommendation"] or ""])

with open(out_path("documents.csv"), "w", encoding="utf-8", newline="") as fh:
    rows = list(doc_rows.values())
    w = csv.DictWriter(fh, fieldnames=list(rows[0]) if rows else ["url"])
    w.writeheader()
    w.writerows(rows)

print("Wrote:")
for name in ("fixes.json", "repaired_ummc.json", "review_report.csv", "identifier_summary.csv", "documents.csv"):
    print("  ", out_path(name))

Wrote:
   output\missing_version_fixes.json
   output\missing_version_repaired_ummc.json
   output\missing_version_review_report.csv
   output\missing_version_identifier_summary.csv
   output\missing_version_documents.csv


## Step 8 — Lineage report

In [16]:
def lineage_report_md():
    conf = Counter(f["confidence"] for f in all_fixes)
    lines = [f"# {OUTPUT_PREFIX.replace('_', ' ').title()} — lineage repair report",
             "", f"Generated {datetime.now(timezone.utc):%Y-%m-%d %H:%M UTC} from `{os.path.basename(INPUT_FILE)}` "
                 f"following *Collection_Lineage_via_ATBD_Skill*.",
             "", f"* Collections: {len(records)}  · associations: {len(all_fixes)}",
             f"* Resolved: {sum(1 for f in all_fixes if f['new_association'])}  · unresolved: "
             f"{sum(1 for f in all_fixes if not f['new_association'])}",
             f"* Confidence: HIGH {conf['HIGH']} · MEDIUM {conf['MEDIUM']} · LOW {conf['LOW']}",
             f"* Documents fetched: {len(doc_rows)} (valid for lineage: "
             f"{sum(1 for d in doc_rows.values() if d['valid_for_lineage'])})", ""]
    by_cid = defaultdict(list)
    for f in all_fixes:
        by_cid[f["concept_id"]].append(f)
    for wr in wrappers:
        fs = by_cid[wr["conceptId"]]
        lines += [f"## {wr['collection_shortName']} v{wr['collection_version']} (`{wr['conceptId']}`)", ""]
        for d in wr["documents"]:
            lines.append(f"* Document (priority {d['priority']}, {'valid' if d['valid_for_lineage'] else 'not valid'}): {d['url']}")
        if not wr["documents"]:
            lines.append("* No lineage document linked in CMR or the input file")
        lines += ["", "| Type | Original | Repaired | Basis | Confidence | Note |", "|---|---|---|---|---|---|"]
        for f in fs:
            before = f"{f['original_id']} v{f['original_version'] or '—'}"
            after = f"{f['short_name']} v{f['version']}" if f["new_association"] else f"*{f['status']}*"
            basis = f"{f['id_basis'] or '—'} / {f['version_basis'] or '—'}"
            note = (f["confidence_reason"] or "").replace("|", "/")
            lines.append(f"| {f['type']} | {before} | {after} | {basis} | {f['confidence']} | {note} |")
        quotes = [f for f in fs if f["lineage_statement"]]
        for f in quotes[:3]:
            st = f["lineage_statement"]
            tag = "" if st.get("exact") else " *(approximate text extraction)*"
            lines.append(f"\n> **{f['original_id']}** — {st['doc']}"
                         f"{', p. ' + str(st['page']) if st.get('page') else ''}{tag}: \"{st['quote'][:400]}\"")
        lines.append("")
    return "\n".join(lines)


with open(out_path("lineage_report.md"), "w", encoding="utf-8") as fh:
    fh.write(lineage_report_md())
print("Wrote", out_path("lineage_report.md"))

Wrote output\missing_version_lineage_report.md


## Summary and review queue

In [17]:
print("Status:       ", dict(Counter(f["status"] for f in all_fixes).most_common()))
print("Confidence:   ", dict(Counter(f["confidence"] for f in all_fixes).most_common()))
print("Id basis:     ", dict(Counter(str(f["id_basis"]) for f in all_fixes).most_common()))
print("Version basis:", dict(Counter(str(f["version_basis"]) for f in all_fixes).most_common()))
print("Level check:  ", dict(Counter((f["cmr_validation"] or {}).get("level_progression") for f in all_fixes).most_common()))
print("With document quote:", sum(1 for f in all_fixes if f["lineage_statement"]))
print("Records with associations still invalid under the new rules:",
      sum(1 for w in wrappers if w["remaining_invalid"]))

queue = [f for f in all_fixes if f["confidence"] != "HIGH"]
print(f"\nReview queue ({len(queue)} associations, grouped by original identifier):")
for oid, n in Counter(f["original_id"] for f in queue).most_common(40):
    f = next(x for x in queue if x["original_id"] == oid)
    target = f"{f['short_name']} v{f['version']}" if f["new_association"] else f["status"]
    print(f"  {n:>4} x {oid:<34} -> {target:<38} {f['confidence']:<6} {f['confidence_reason'][:70]}")

Status:        {'resolved': 139}
Confidence:    {'HIGH': 133, 'MEDIUM': 6}
Id basis:      {'exact': 126, 'case_normalized': 4, 'embedded_verified': 3, 'typo_corrected': 2, 'embedded_split': 2, 'standard_counterpart': 1, 'documented_companion': 1}
Version basis: {'unique_in_cmr': 136, 'original_verified': 3}
Level check:   {'not_applicable': 116, 'ok': 23}
With document quote: 9
Records with associations still invalid under the new rules: 0

Review queue (6 associations, grouped by original identifier):
     2 x VNP46A1_1                          -> VNP46A1 v2                             MEDIUM identifier: embedded_split; document evidence
     1 x MOD09A1N                           -> MOD09A1 v061                           MEDIUM identifier: standard_counterpart
     1 x MOD09CMA                           -> MOD09CMG v061                          MEDIUM identifier: documented_companion; document evidence
     1 x SENIINEL-1C_SLC                    -> SENTINEL-1C_SLC v1                 